# Varredura v3 — 10 folds, dois mapas, saida em painel

Muda **um** parametro em relacao a v2: `N_FOLDS` de 4 para 10. Na v2 os quatro vereditos de
equivalencia sairam inconclusivos porque o IC vinha de 4 folds; com 1.315 falantes o corpus
suporta 10 folds sem quebrar a independencia de sujeito, e o IC estreita o bastante para
decidir. Folds sao a unica alavanca que adiciona informacao independente aqui.

O resto e' identico a v2, incluindo sementes — a metade `zz` deve reproduzir os numeros que
voce ja tem.

Duas coisas de uma vez:

**(1) Diagnostico corrigido.** O teste da v1 estava mal desenhado. Ele correlacionava residuo
com AUC via Spearman, mas as duas grandezas nao tem essa geometria: o residuo **cai
monotonicamente** com `c`, enquanto o AUC e' **unimodal** (pico em c ~ 0.07). Correlacao
monotona nao mede monotona-contra-unimodal, e por isso saiu rho = -0.244.

Pior: o residuo de Frobenius e' **degenerado no limite c -> 0**. Ali a Gram vira quase toda de
uns e *qualquer* kernel a aproxima — dai o minimo de residuo em c=0.01 justamente onde o AUC
e' ruim. O residuo so mede classicalidade fora dessa regiao. A secao 5 filtra os pontos
degenerados por `gram_off_sd` e separa o ramo descendente antes de testar.

**(2) Mapa sem emaranhamento.** A mesma grade rodada com `entangle=False`. Se as duas
superficies coincidirem, a afirmacao do artigo sobe de "esse mapa ZZ nao ajuda" para
"a estrutura de emaranhamento e' irrelevante nesse regime" — que e' bem mais forte e bem mais
dificil de refutar.

Isso e' uma alegacao de **equivalencia**, nao de diferenca. Teste de significancia nao serve:
nao rejeitar H0 nao prova igualdade. A secao 6 usa TOST com margem pre-fixada em 0.01 de AUC.

In [ ]:
# !pip install pennylane -q
import numpy as np, pandas as pd, time
import matplotlib.pyplot as plt
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, t as tdist
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR, DIM = "#0d0d0d", "#e8e8e8", "#ff6b00", "#8a8a8a"
FONT = "'JetBrains Mono','Fira Code',monospace"

# Painel escuro com acento laranja no lugar de print cru.
def painel(titulo, linhas, tabela=None, destaque=None):
    corpo = "".join(
        f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
        for l in linhas)
    tab = ""
    if tabela is not None:
        sty = [("border-collapse", "collapse"), ("font-family", FONT),
               ("font-size", "12px"), ("color", FG), ("margin", ".8em 0 .2em 0")]
        tab = (tabela.style.hide(axis="index")
               .set_table_styles([
                   {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                                ("border-bottom", f"1px solid {OR}"),
                                                ("padding", "4px 12px")]},
                   {"selector": "td", "props": [("text-align", "right"),
                                                ("padding", "3px 12px"),
                                                ("border-bottom", f"1px solid #222")]}])
               .set_table_attributes(f"style='{';'.join(f'{k}:{v}' for k, v in sty)}'")
               .to_html())
    dest = ""
    if destaque:
        dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
                f"background:#1a1206;color:{FG}'>{destaque}</div>")
    display(HTML(
        f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
        f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
        f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
        f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

In [ ]:
PKL_PATH = "features_v5.pkl"
META = ["speaker_id", "session_id", "label", "diagnosis", "vowel", "pitch", "wav"]

feat = pd.read_pickle(PKL_PATH)
FEATS = [col for col in feat.columns if col not in META]
X_all = feat[FEATS].values.astype(float)
y_all = feat["label"].values.astype(int)
groups_all = feat["speaker_id"].values
assert not set(FEATS) & set(META), 'metadado vazando como feature'
painel("dados", [
    f"{X_all.shape[0]} gravacoes &nbsp;&middot;&nbsp; {len(FEATS)} features "
    f"&nbsp;&middot;&nbsp; {pd.unique(groups_all).size} falantes",
    f"prevalencia = {y_all.mean():.3f}",
    "metadados excluidos: " + ", ".join(META[1:])])

In [ ]:
def _state_fn(n_qubits, reps, entangle):
    dev = qml.device("default.qubit", wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x):
        for _ in range(reps):
            for i in range(n_qubits):
                qml.Hadamard(wires=i)
                qml.RZ(2.0 * x[..., i], wires=i)
            if entangle:
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                    qml.RZ(2.0 * (np.pi - x[..., i]) * (np.pi - x[..., i + 1]), wires=i + 1)
                    qml.CNOT(wires=[i, i + 1])
        return qml.state()
    return circuit

def states(X, n_qubits, reps=2, entangle=True, batch=512):
    f = _state_fn(n_qubits, reps, entangle)
    return np.vstack([np.asarray(f(X[s:s + batch])) for s in range(0, len(X), batch)])

def gram(A, B):
    return np.abs(A @ B.conj().T) ** 2

def fit_surrogate(K_q, Xa):
    D2 = ((Xa[:, None, :] - Xa[None, :, :]) ** 2).sum(-1)
    loss = lambda lg: np.linalg.norm(K_q - np.exp(-np.exp(lg) * D2))
    r = minimize_scalar(loss, bounds=(-8, 6), method="bounded")
    return float(np.exp(r.x)), float(r.fun) / np.sqrt(K_q.size)

def align(K, y):
    Y = np.outer(2 * y - 1, 2 * y - 1)
    return float((K * Y).sum() / (np.linalg.norm(K) * np.linalg.norm(Y)))

def off_diag_sd(K):
    iu = np.triu_indices_from(K, k=1)
    return float(K[iu].std())

def subsample(ytr, gtr, n_per_class, rng):
    idx = []
    for cls in (0, 1):
        spk = rng.permutation(np.unique(gtr[ytr == cls]))
        take = []
        for s in spk:
            take += list(np.where((gtr == s) & (ytr == cls))[0])
            if len(take) >= n_per_class:
                break
        idx += take[:n_per_class]
    return np.array(idx)

## 3. Grade

Mesma grade da v1, agora rodada para os **dois** mapas. Sementes identicas as da v1
(`random_state=7`, `default_rng(1000*fold + rep)`), entao a metade `zz` deve reproduzir os
numeros que voce ja tem — se nao reproduzir, ha algo errado e vale parar antes de seguir.

In [ ]:
QUBITS = [4, 6, 8, 10]
SCALES = np.round(np.geomspace(0.01, 2.0, 12), 4)
MAPAS = [("zz", True), ("prod", False)]
N_PER_CLASS = 200
N_FOLDS, N_REPS = 10, 3
C_SVM = 1.0
REPS_CIRCUITO = 2
MARGEM_EQUIV = 0.01          # margem de equivalencia em AUC, fixada ANTES de rodar

n_pontos = len(QUBITS) * len(SCALES) * len(MAPAS) * N_FOLDS * N_REPS
painel("orcamento da grade", [
    f"{len(QUBITS)} qubits &times; {len(SCALES)} bandwidths &times; {len(MAPAS)} mapas "
    f"&times; {N_FOLDS} folds &times; {N_REPS} reps",
    f"= {n_pontos} avaliacoes de kernel",
    f"estimativa: {n_pontos*2.7/60:.0f}-{n_pontos*4/60:.0f} min em CPU",
    f"margem de equivalencia fixada em &plusmn;{MARGEM_EQUIV} de AUC"],
    destaque="Nada e' reduzido automaticamente. Ajuste QUBITS / SCALES / N_REPS / "
             "N_FOLDS como preferir — so folds adicionam informacao independente.")

In [ ]:
rows = []
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=7)
t0 = time.time()

for fold, (tr, te) in enumerate(sgkf.split(X_all, y_all, groups_all)):
    for rep in range(N_REPS):
        rng = np.random.default_rng(1000 * fold + rep)
        sub = subsample(y_all[tr], groups_all[tr], N_PER_CLASS, rng)
        ytr, yte = y_all[tr][sub], y_all[te]
        imp = SimpleImputer(strategy="median").fit(X_all[tr][sub])
        Xtr_i, Xte_i = imp.transform(X_all[tr][sub]), imp.transform(X_all[te])

        for q in QUBITS:
            sel = SelectKBest(f_classif, k=q).fit(Xtr_i, ytr)
            sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr_i))
            A = sc.transform(sel.transform(Xtr_i))
            B = np.clip(sc.transform(sel.transform(Xte_i)), 0, np.pi)

            for cval in SCALES:
                Ac, Bc = A * cval, B * cval
                for nome, ent in MAPAS:
                    S1 = states(Ac, q, REPS_CIRCUITO, ent)
                    S2 = states(Bc, q, REPS_CIRCUITO, ent)
                    K1 = gram(S1, S1)
                    m = SVC(kernel="precomputed", C=C_SVM).fit(K1, ytr)
                    auc_q = roc_auc_score(yte, m.decision_function(gram(S2, S1)))
                    g_s, resid = fit_surrogate(K1, Ac)
                    ms = SVC(kernel="rbf", gamma=g_s, C=C_SVM).fit(Ac, ytr)
                    rows.append(dict(mapa=nome, fold=fold, rep=rep, n_qubits=q, c=cval,
                                     auc_q=auc_q,
                                     auc_surr=roc_auc_score(yte, ms.decision_function(Bc)),
                                     resid=resid, align=align(K1, ytr),
                                     gram_off_sd=off_diag_sd(K1), gamma_surr=g_s))
    print(f"fold {fold} ok — {time.time() - t0:.0f}s")

df = pd.DataFrame(rows)
agg = df.groupby(["mapa", "n_qubits", "c"]).mean(numeric_only=True).reset_index()
zz = agg[agg.mapa == "zz"]
piv = zz.pivot(index="c", columns="n_qubits", values="auc_q").round(4).reset_index()
piv.columns = ["c"] + [f"{q}q" for q in piv.columns[1:]]
painel("AUC do mapa ZZ por bandwidth e qubits",
       [f"grade concluida em {time.time()-t0:.0f}s",
        f"melhor celula: AUC={zz.auc_q.max():.4f}"],
       tabela=piv)

## 4. As duas superficies

Cada painel e' um numero de qubits. Linha cheia = ZZ (com emaranhamento), tracejada = produto
(sem). Se as curvas se sobrepuserem em toda a faixa de bandwidth, o emaranhamento nao esta
fazendo trabalho nenhum.

In [ ]:
fig, ax = plt.subplots(1, len(QUBITS), figsize=(4 * len(QUBITS), 3.6), sharey=True)
for k, q in enumerate(QUBITS):
    a = ax[k] if len(QUBITS) > 1 else ax
    for nome, cor, ls in (("zz", "#2a6f97", "-"), ("prod", "#c1121f", "--")):
        g = agg[(agg.mapa == nome) & (agg.n_qubits == q)].sort_values("c")
        a.semilogx(g.c, g.auc_q, ls, color=cor, marker="o", ms=3.5, label=nome)
    a.set(xlabel="bandwidth c", title=f"{q} qubits"); a.grid(alpha=.3)
    if k == 0:
        a.set_ylabel("AUC"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Diagnostico corrigido do mecanismo

Tres testes, nesta ordem de importancia:

1. **`auc_q` vs `auc_surr`** — o teste primario. Se andam juntos, sao o mesmo modelo.
2. **`resid` vs `auc_q` fora da regiao degenerada** — o teste que a v1 fez errado.
3. **concentracao no ramo descendente** — explica a queda para `c` grande.

In [ ]:
zz = agg[agg.mapa == "zz"]

piso = zz.gram_off_sd.max() * 0.05
nd = zz[zz.gram_off_sd > piso]
r1, p1 = spearmanr(zz.auc_q, zz.auc_surr)
dif = (zz.auc_q - zz.auc_surr).abs()
r2, p2 = spearmanr(nd.resid, nd.auc_q)
desc = pd.concat([g[g.c >= g.loc[g.auc_q.idxmax(), "c"]]
                  for _, g in zz.groupby("n_qubits")])
r3, p3 = spearmanr(desc.gram_off_sd, desc.auc_q)

msg = []
if r1 > 0.9 and dif.max() < 0.05:
    msg.append("O kernel quantico acompanha uma gaussiana classica em toda a grade.")
if r2 < -0.4 and p2 < 0.05:
    msg.append("Fora da degenerescencia, melhor desempenho onde a aproximacao classica "
               "e' melhor.")
elif r2 > 0.4 and p2 < 0.05:
    msg.append("ATENCAO: melhor desempenho onde a aproximacao classica FALHA — contradiz "
               "a auditoria.")
elif abs(r2) < 0.4:
    msg.append("O residuo de Frobenius nao discrimina aqui: e' degenerado em c&rarr;0, onde "
               "a Gram vira quase toda de uns. Tratar como diagnostico secundario.")
if p3 < 0.05 and r3 > 0.4:
    msg.append("A queda para bandwidth grande <b>e'</b> concentracao exponencial da Gram.")

painel("mecanismo", [
    f"pontos nao degenerados: {len(nd)}/{len(zz)}",
    f"[1] Spearman(auc_q, auc_surr) = <b>{r1:+.3f}</b> (p={p1:.2g}) &nbsp;|&nbsp; "
    f"|dif| mediana={dif.median():.4f} max={dif.max():.4f}",
    f"[2] Spearman(resid, auc_q) fora da degenerescencia = {r2:+.3f} (p={p2:.2g})",
    f"[3] Spearman(gram_off_sd, auc_q) no ramo descendente = <b>{r3:+.3f}</b> (p={p3:.2g})"],
    destaque="<br>".join(msg))

## 6. Equivalencia dos dois mapas (TOST)

"Nao rejeitei H0" nao prova igualdade. Para afirmar que o emaranhamento e' irrelevante e'
preciso um teste de **equivalencia**: mostrar que o IC da diferenca cabe inteiro dentro de
+/- `MARGEM_EQUIV`.

A unidade de analise e' o **fold**, nao a linha da grade: as repeticoes reamostram dentro dos
mesmos folds e as 48 celulas da grade compartilham os mesmos folds. Tratar tudo como
independente seria a mesma pseudorreplicacao que ja apareceu nos subgrupos.

In [ ]:
pf = df.pivot_table(index=["n_qubits", "c", "fold"], columns="mapa",
                    values="auc_q").reset_index()
pf["delta"] = pf["zz"] - pf["prod"]

linhas = []
for q, g in pf.groupby("n_qubits"):
    porfold = g.groupby("fold").delta.mean()          # unidade = fold
    n = len(porfold); mu = porfold.mean()
    se = porfold.std(ddof=1) / np.sqrt(n)
    tc = tdist.ppf(0.975, n - 1)
    lo, hi = mu - tc * se, mu + tc * se
    if (lo > -MARGEM_EQUIV) and (hi < MARGEM_EQUIV):
        vered = "equivalente"
    elif (hi < -MARGEM_EQUIV) or (lo > MARGEM_EQUIV):
        vered = "diferenca real"
    else:
        vered = "inconclusivo"
    linhas.append(dict(qubits=q, folds=n, delta=round(mu, 4), ic95_lo=round(lo, 4),
                       ic95_hi=round(hi, 4), veredito=vered))
res = pd.DataFrame(linhas)

dentro = (pf.groupby(["n_qubits", "c"]).delta.mean().abs() < MARGEM_EQUIV).mean()
r_map, p_map = spearmanr(agg[agg.mapa == "zz"].auc_q.values,
                         agg[agg.mapa == "prod"].auc_q.values)

eq = (res.veredito == "equivalente"); dfr = (res.veredito == "diferenca real")
if eq.all():
    dest = ("<b>EQUIVALENCIA ESTABELECIDA</b> em todos os numeros de qubits: a estrutura de "
            "emaranhamento e' irrelevante nesse regime. Essa e' a afirmacao forte do artigo.")
elif dfr.any():
    lado = "ABAIXO" if res[dfr].delta.mean() < 0 else "ACIMA"
    dest = (f"<b>DIFERENCA REAL</b> em {list(res[dfr].qubits)}: o mapa ZZ fica {lado} do "
            "produto por mais que a margem. Nao e' equivalencia — e' o emaranhamento "
            "atrapalhando. Afirmacao mais forte que a planejada, mas exige reescrever "
            "a hipotese.")
elif eq.any():
    dest = f"Equivalencia so em {list(res[eq].qubits)}. Reporte por numero de qubits."
else:
    dest = ("<b>INCONCLUSIVO</b>: o IC ainda atravessa a margem mesmo com "
            f"{N_FOLDS} folds. Falta poder — aumentar N_FOLDS adiciona informacao "
            "independente; N_REPS nao.")

painel("equivalencia dos dois mapas (TOST)", [
    f"margem fixada antes de rodar: &plusmn;{MARGEM_EQUIV} de AUC",
    f"unidade de analise: fold (n={N_FOLDS}), nao linha da grade",
    f"celulas com |delta| &lt; {MARGEM_EQUIV}: <b>{dentro:.0%}</b>",
    f"Spearman(auc_zz, auc_prod) = {r_map:+.3f} (p={p_map:.2g})"],
    tabela=res, destaque=dest)

## 7. O que isso da ao artigo

- Secao 4: figura de duas superficies — sustenta a afirmacao sobre emaranhamento sem uma linha de texto.
- Secao 5: o mecanismo, agora medido com a geometria certa.
- Secao 6: equivalencia com margem pre-registrada, que e' o padrao que um referee metodologico espera.

E o argumento que fecha a discussao: **a auditoria nao precisa de hardware real**. Voce nao esta
reivindicando vantagem quantica — esta mostrando que ela nao aparece sob controle adequado, e a
simulacao de statevector e' exata. Isso neutraliza a objecao que derrubaria um artigo de
proposta de arquitetura.